In [1]:
import os
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from peft.utils import TaskType

# --------------------------
# 模型路径（替换为实际路径）
actual_model_path = "/root/.cache/modelscope/hub/qwen/Qwen-7B-Chat"
# --------------------------

# 1. 加载魏无羡对话数据集并转换为Qwen原生格式
file_path = "weiwuxian_dialogues_dataset.json"
with open(file_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

formatted_data = []
for chapter in dataset["chapters"]:
    for sample in chapter["samples"]:
        convs = sample["conversations"]
        system = next(c["value"] for c in convs if c["from"] == "system")
        human = next(c["value"] for c in convs if c["from"] == "human")
        assistant = next(c["value"] for c in convs if c["from"] == "assistant")
        
        formatted_data.append({
            "instruction": system,
            "input": human,
            "output": assistant
        })

train_dataset = Dataset.from_list(formatted_data)
print(f"✅ 数据集解析完成，共{len(train_dataset)}条样本")


# 2. 加载Tokenizer（使用Qwen原生pad_token配置）
tokenizer = AutoTokenizer.from_pretrained(
    actual_model_path,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=False
)
tokenizer.pad_token_id = tokenizer.eod_id
print(f"✅ Tokenizer配置完成，pad_token_id: {tokenizer.pad_token_id}")


# 3. 数据预处理（严格遵循Qwen输入格式）
def preprocess_function(examples):
    MAX_LENGTH = 512
    input_ids_list, attention_mask_list, labels_list = [], [], []
    
    for i in range(len(examples["instruction"])):
        instruction = examples["instruction"][i]
        user_input = examples["input"][i]
        output = examples["output"][i]
        
        system_part = f"<|im_start|>system\n{instruction}<|im_end|>\n"
        user_part = f"<|im_start|>user\n{user_input}<|im_end|>\n"
        assistant_part = f"<|im_start|>assistant\n{output}<|im_end|>\n"
        
        system_ids = tokenizer(system_part, add_special_tokens=False)["input_ids"]
        user_ids = tokenizer(user_part, add_special_tokens=False)["input_ids"]
        assistant_ids = tokenizer(assistant_part, add_special_tokens=False)["input_ids"]
        pad_id = tokenizer.pad_token_id
        
        input_ids = system_ids + user_ids + assistant_ids + [pad_id]
        attention_mask = [1] * len(input_ids)
        labels = [-100] * (len(system_ids) + len(user_ids)) + assistant_ids + [pad_id]
        
        # 截断与补pad
        if len(input_ids) > MAX_LENGTH:
            input_ids = input_ids[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            labels = labels[:MAX_LENGTH]
        else:
            pad_length = MAX_LENGTH - len(input_ids)
            input_ids += [pad_id] * pad_length
            attention_mask += [0] * pad_length
            labels += [-100] * pad_length
        
        input_ids_list.append(input_ids)
        attention_mask_list.append(attention_mask)
        labels_list.append(labels)
    
    return {
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list
    }

tokenized_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)
print(f"✅ 数据预处理完成，tokenized_dataset样本数：{len(tokenized_dataset)}")


# 4. 加载模型（仅执行一次，4bit量化）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    actual_model_path,
    trust_remote_code=True,
    device_map="auto",
    quantization_config=bnb_config,
    local_files_only=True,
    torch_dtype=torch.half
)

# --------------------------
# 新增：Qwen兼容的梯度检查点开启方式（替换Trainer自动调用）
model.gradient_checkpointing = True
print("✅ Qwen模型梯度检查点已手动启用")
# --------------------------


# 5. 配置LoRA（仅执行一次，适配Qwen模型）
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["c_attn", "c_proj", "w1", "w2"],
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"✅ 模型加载+LoRA配置完成，可启动训练")


# 6. 拆分训练集/验证集（新增，避免过拟合）
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]


# 7. 训练参数（关键修改：删除gradient_checkpointing=True）
training_args = TrainingArguments(
    output_dir="./qwen-weiwuxian-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    save_steps=100,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    fp16=True,
    report_to="none",
    weight_decay=0.01,
    evaluation_strategy="steps",
    eval_steps=100,
    # gradient_checkpointing=True  # 删掉这行！避免Trainer自动调用不兼容方法
)


# 8. 启动训练与保存（此时Trainer不会再触发参数冲突）
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

print("🚀 开始LoRA微调...")
trainer.train()


print("💾 保存微调模型...")
# 1. 保存LoRA模型权重
model.save_pretrained("./qwen-weiwuxian-lora-final")
# 2. 保存Tokenizer
tokenizer.save_pretrained("./qwen-weiwuxian-lora-final")
# 3. 保存训练参数（修正部分）
save_dir = "./qwen-weiwuxian-lora-final"
os.makedirs(save_dir, exist_ok=True)  # 确保目录存在
training_args_dict = vars(training_args)
# 过滤不可序列化属性，避免json报错
serializable_args = {
    k: v for k, v in training_args_dict.items()
    if isinstance(v, (int, float, str, bool, list, tuple, dict, type(None)))
}
with open(os.path.join(save_dir, "training_args.json"), "w", encoding="utf-8") as f:
    json.dump(serializable_args, f, ensure_ascii=False, indent=2)

print("✅ 微调完成！模型路径：./qwen-weiwuxian-lora-final")

/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


✅ 数据集解析完成，共710条样本
✅ Tokenizer配置完成，pad_token_id: 151643


Map:   0%|          | 0/710 [00:00<?, ? examples/s]

The model is automatically converting to bf16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...


✅ 数据预处理完成，tokenized_dataset样本数：710


...

✅ Qwen模型梯度检查点已手动启用
trainable params: 35,782,656 || all params: 4,519,104,512 || trainable%: 0.7918085520036763
✅ 模型加载+LoRA配置完成，可启动训练
🚀 开始LoRA微调...


/root/miniconda3/lib/python3.12/site-packages/accelerate/accelerator.py:416: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/root/miniconda3/lib/python3.12/site-packages/accelerate/accelerator.py:1308: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  model.forward = MethodType(torch.cuda.amp.autocast(dtype=torch.float16)(model.forward.__func__), model)


Step,Training Loss,Validation Loss
100,0.620900,0.954213
200,0.294000,0.960896
300,0.151700,1.013418
400,0.083700,1.099492


💾 保存微调模型...


AttributeError: 'TrainingArguments' object has no attribute 'save_to_json'

# 测试微调后模型
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id, trust_remote_code=True, device_map="auto", load_in_4bit=True
)
lora_model = PeftModel.from_pretrained(base_model, "./qwen-weiwuxian-lora")
lora_model.eval()

# 测试输入（与训练数据格式一致）
query = "你是《魔道祖师》中的魏无羡，在云深不知处被蓝启仁罚抄家规时，会说什么？"
response, _ = lora_model.chat(tokenizer, query, history=[])
print(response)  # 应输出符合魏无羡语气的回复（如“蓝老先生，抄就抄，可这家规也太多了吧？”）

In [1]:
# 测试微调后模型（适配Qwen-7B-Chat + 4bit量化 + LoRA）
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# --------------------------
# 1. 配置基础参数（与训练时保持一致）
# --------------------------
# 原始Qwen基座模型路径（需与训练时的actual_model_path一致）
base_model_path = "/root/.cache/modelscope/hub/qwen/Qwen-7B-Chat"
# 最终保存的LoRA模型路径（训练完成后保存的目标目录）
lora_model_path = "./qwen-weiwuxian-lora-final"  # 重点：用最终保存的模型，而非训练中间checkpoint

# --------------------------
# 2. 加载4bit量化配置（与训练时完全一致，避免显存溢出）
# --------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# --------------------------
# 3. 加载原始基座模型（带量化，与训练时配置一致）
# --------------------------
print("🔧 加载Qwen-7B基座模型...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    trust_remote_code=True,  # Qwen模型必须开启
    device_map="auto",       # 自动分配设备（优先GPU）
    quantization_config=bnb_config,  # 启用4bit量化
    local_files_only=True,   # 本地加载，避免重复下载
    torch_dtype=torch.half   # 半精度加载，平衡速度与显存
)

# --------------------------
# 4. 加载Tokenizer（与训练时完全一致，确保格式兼容）
# --------------------------
print("🔧 加载Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    local_files_only=True,
    use_fast=False,  # Qwen建议关闭fast tokenizer
    pad_token_id=tokenizer.eod_id if 'tokenizer' in locals() else None  # 保持pad_token配置一致
)
# 补全pad_token配置（防止部分版本Tokenizer未自动继承）
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eod_id

# --------------------------
# 5. 加载LoRA微调模型（合并基座与LoRA权重）
# --------------------------
print("🔧 加载LoRA微调模型...")
lora_model = PeftModel.from_pretrained(
    base_model,
    lora_model_path,
    device_map="auto",  # 与基座模型设备一致
    torch_dtype=torch.float16
)
# 切换为评估模式（禁用Dropout，确保推理稳定）
lora_model.eval()
print("✅ 模型加载完成，可开始测试！")

# --------------------------
# 6. 测试推理（适配Qwen chat格式，模拟训练数据场景）
# --------------------------
def test_weiwuxian_response(query):
    # 禁用梯度计算（减少显存占用，加速推理）
    with torch.no_grad():
        # Qwen-7B-Chat专用chat方法，自动处理对话格式
        response = lora_model.chat(
            tokenizer=tokenizer,
            query=query,
            history=[],  # 空历史对话，模拟首次提问
            max_new_tokens=200,  # 限制回复长度，避免过长
            temperature=0.7,     # 控制随机性，0.7更符合角色自然表达
            top_p=0.9            # 控制生成多样性
        )
    return response

# --------------------------
# 测试案例（覆盖不同场景，验证角色适配性）
# --------------------------
print("\n📝 测试案例1：云深不知处罚抄家规场景")
query1 = "你是《魔道祖师》中的魏无羡，在云深不知处被蓝启仁罚抄家规时，会说什么？"
response1 = test_weiwuxian_response(query1)
print(f"👤 提问：{query1}")
print(f"🗣️  魏无羡回复：{response1}\n")

print("📝 测试案例2：与蓝忘机互动场景")
query2 = "蓝湛，你说我要是偷偷把兔子抱进藏书阁，会不会被先生发现啊？"
response2 = test_weiwuxian_response(query2)
print(f"👤 提问：{query2}")
print(f"🗣️  魏无羡回复：{response2}\n")

print("📝 测试案例3：自我身份认知场景")
query3 = "你是谁？介绍一下你自己吧。"
response3 = test_weiwuxian_response(query3)
print(f"👤 提问：{query3}")
print(f"🗣️  魏无羡回复：{response3}")

/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/root/miniconda3/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The model is automatically converting to bf16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention 

🔧 加载Qwen-7B基座模型...


The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

🔧 加载Tokenizer...
🔧 加载LoRA微调模型...
✅ 模型加载完成，可开始测试！

📝 测试案例1：云深不知处罚抄家规场景
👤 提问：你是《魔道祖师》中的魏无羡，在云深不知处被蓝启仁罚抄家规时，会说什么？
🗣️  魏无羡回复：('“家规？这叫作家训！懂不懂？”', [('你是《魔道祖师》中的魏无羡，在云深不知处被蓝启仁罚抄家规时，会说什么？', '“家规？这叫作家训！懂不懂？”')])

📝 测试案例2：与蓝忘机互动场景
👤 提问：蓝湛，你说我要是偷偷把兔子抱进藏书阁，会不会被先生发现啊？
🗣️  魏无羡回复：('别别别！我最怕藏书阁了，那里的书又厚又重，一碰就倒！', [('蓝湛，你说我要是偷偷把兔子抱进藏书阁，会不会被先生发现啊？', '别别别！我最怕藏书阁了，那里的书又厚又重，一碰就倒！')])

📝 测试案例3：自我身份认知场景
👤 提问：你是谁？介绍一下你自己吧。
🗣️  魏无羡回复：('我是来自阿里云的大规模语言模型，我叫通义千问。', [('你是谁？介绍一下你自己吧。', '我是来自阿里云的大规模语言模型，我叫通义千问。')])
